# 4.21 Manifold learning: Isomap, LLE, MDS, Laplacian eigenmaps

Manifold learning turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise.

Part 4 moves from prediction with labels to discovery without labels. Manifold methods replace straight-line distance with neighborhood graphs, geodesics, or local reconstruction. The same D1-D5 ladder lets us compare Isomap, LLE, MDS, and Laplacian eigenmaps without changing the data interface.

Save a copy to Drive to edit.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.decomposition import NMF
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.manifold import MDS
from sklearn.manifold import SpectralEmbedding
from sklearn.manifold import TSNE
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.random_projection import GaussianRandomProjection
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

def dimred_ladder():
    """D1..D5 dimensionality-reduction ladder. Returns [(name, X, y), ...] of rising ambient dim.

    2-D toy -> 3-D swiss-roll-ish -> digits(64-D) -> the same with noise dims -> a wide feature set.
    y is a color/label for visualization only.
    """
    rungs = []
    rng = np.random.default_rng(3)

    t = np.linspace(0, 4, 120)
    x1 = np.column_stack([t, 0.5 * t + rng.normal(0, 0.05, 120)])
    rungs.append(("D1 near-1-D line in 2-D", x1, t))

    tt = np.linspace(0, 3 * np.pi, 200)
    x2 = np.column_stack([tt * np.cos(tt), 8 * rng.random(200), tt * np.sin(tt)])
    rungs.append(("D2 swiss-roll (3-D)", x2, tt))

    digits = load_digits()
    rungs.append(("D3 digits (real, 64-D)", digits.data / 16.0, digits.target))

    xn = np.hstack([digits.data / 16.0, rng.normal(0, 1, size=(digits.data.shape[0], 32))])
    rungs.append(("D4 digits + 32 noise dims", xn, digits.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (30-D)", bc.data, bc.target))

    return rungs



def sample_for_embedding(X, y, max_points=500, seed=SEED):
    rng = np.random.default_rng(seed)
    if X.shape[0] <= max_points:
        return X, y
    idx = rng.choice(X.shape[0], size=max_points, replace=False)
    order = np.sort(idx)
    return X[order], y[order]


def standardize_for_geometry(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)


def nonnegative_for_factorization(X):
    scaler = MinMaxScaler()
    return scaler.fit_transform(X)


def safe_trustworthiness(X, Z):
    neighbors = min(10, max(1, (X.shape[0] - 1) // 3))
    return float(trustworthiness(X, Z, n_neighbors=neighbors))


def plot_ladder_embeddings(results, metric_name):
    fig, axes = plt.subplots(1, len(results), figsize=(17, 3.4))
    for ax, item in zip(axes, results):
        scatter = ax.scatter(item["Z"][:, 0], item["Z"][:, 1], c=item["y"], s=10, cmap="viridis")
        ax.set_title(item["name"].split("(")[0], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle("2D embeddings across the D1-D5 ladder")
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 3.5))
    xs = np.arange(1, len(results) + 1)
    ys = [item["metric"] for item in results]
    ax.plot(xs, ys, marker="o")
    ax.set_xticks(xs)
    ax.set_xticklabels([f"D{i}" for i in xs])
    ax.set_ylabel(metric_name)
    ax.set_xlabel("dataset rung")
    ax.set_title(f"{metric_name} vs. ladder complexity")
    ax.grid(True, alpha=0.3)
    plt.show()


def preview_ladder(rungs):
    for i, (name, X, y) in enumerate(rungs, start=1):
        unique = np.unique(y)
        label_info = len(unique) if unique.size < 30 else "continuous"
        print(f"D{i}: {name} | X={X.shape} | color/label info={label_info}")
        print(np.round(X[:3, :min(5, X.shape[1])], 3))

## The concept, built once: compare graph/manifold algorithms

The shared graph audit is $$L=D-A,\qquad Lv=\lambda v$$. The eigenvalues $[0,0,2,2]$ tell us the toy graph has two disconnected pieces before any embedding is trusted.

In [ ]:
A = np.array([[0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 1.0, 0.0]])
D = np.diag(A.sum(axis=1))
L = D - A
eigenvalues = np.linalg.eigvalsh(L)
print("Laplacian")
print(L)
print("eigenvalues", eigenvalues)
assert np.allclose(np.round(eigenvalues, 3), [0.0, 0.0, 2.0, 2.0])

Now build the reusable method. It can call real sklearn Isomap, LLE, MDS, or Laplacian-eigenmap-style `SpectralEmbedding`; the ladder run uses Isomap as the default.

In [ ]:
def method(X, algorithm="isomap", n_neighbors=12, seed=SEED):
    X_scaled = standardize_for_geometry(X)
    neighbors = min(n_neighbors, X_scaled.shape[0] - 1)
    if algorithm == "isomap":
        model = Isomap(n_components=2, n_neighbors=neighbors)
    elif algorithm == "lle":
        model = LocallyLinearEmbedding(n_components=2, n_neighbors=neighbors, random_state=seed, method="standard")
    elif algorithm == "mds":
        model = MDS(n_components=2, random_state=seed, normalized_stress="auto", n_init=1, max_iter=300)
    elif algorithm == "laplacian":
        model = SpectralEmbedding(n_components=2, n_neighbors=neighbors, random_state=seed, affinity="nearest_neighbors")
    else:
        raise ValueError(f"unknown algorithm: {algorithm}")
    return model.fit_transform(X_scaled)

assert method.__name__ == "method

## The dataset ladder

We use the shared F3 dimensionality-reduction ladder: a near-1D toy line, a 3D swiss-roll-style surface, real handwritten digits, digits with added noise dimensions, and a real 30-dimensional breast-cancer feature table. The labels are only colors for visualization; the embedding method does not train on them.

In [ ]:
rungs = dimred_ladder()
preview_ladder(rungs)

## Run the same method across D1-D5

Each rung is standardized or scaled in the same way, embedded into 2D, and scored with trustworthiness. Subsampling is seeded and only bounds future notebook runtime; this build script does not execute the notebook.

In [ ]:
metric_name = "trustworthiness"
results = []
for rung_index, (name, X, y) in enumerate(rungs, start=1):
    X_small, y_small = sample_for_embedding(X, y, max_points=450, seed=SEED + rung_index)
    Z = method(X_small, algorithm="isomap", n_neighbors=12, seed=SEED + rung_index)
    score = safe_trustworthiness(standardize_for_geometry(X_small), Z)
    results.append({"name": name, "X": X_small, "y": y_small, "Z": Z, "metric": score})
    print(f"D{rung_index} | {name:32s} | trustworthiness={score:.3f}")

## Results visualization

The closing figure has two parts: small-multiple embedding panels for D1-D5, then a metric curve as the data become more realistic and higher dimensional.

In [ ]:
plot_ladder_embeddings(results, metric_name)

## Pitfall on the hardest rung: disconnected or short-circuit neighbor graphs

Too few neighbors can disconnect D5; too many can create short circuits that erase manifold geometry. The fix is to sweep neighbors and check stability before trusting one graph.

In [ ]:
name, X, y = rungs[-1]
X_small, y_small = sample_for_embedding(X, y, max_points=450, seed=39)
sweep = []
for neighbors in [3, 6, 12, 24]:
    Z = method(X_small, algorithm="isomap", n_neighbors=neighbors, seed=0)
    score = safe_trustworthiness(standardize_for_geometry(X_small), Z)
    sweep.append((neighbors, score))
print("Isomap neighbor sweep", [(n, round(s, 3)) for n, s in sweep])
print("fix: prefer a stable neighbor range and confirm the graph is not fragmented")

## Evaluate it + Practice

- Report trustworthiness next to a no-skill baseline such as plotting two raw standardized features or a random 2D map.
- Sanity check that nearby points in the original space still have nearby points in the embedding.
- Ablate the key idea: remove scaling, change the random seed, or use too few neighbors/components and watch the metric move.
- Watch for failure signals: unstable layouts, one feature dominating distances, disconnected neighbor graphs, or a better objective with worse held-out structure.
- Treat labels as a post-hoc audit only; unsupervised methods do not get to train on them.


### Practice

Compare Isomap, LLE, MDS, and Laplacian on D2 with the same sample size.

In [ ]:
# Your code here


### Practice

On D1, manually build an adjacency with one bridge edge and recompute the zero eigenvalue count.

In [ ]:
# Your code here


### Practice

Find the neighbor count where D5 trustworthiness stops improving.

In [ ]:
# Your code here
